<a id='any-all'></a>

## 12. 🧩 Pattern 12: any() and all() — Short-Circuit Aggregators — LC 20, 125, 242, 383, 424, 567, 680

---

```
PROBLEM:
  LC 20  — Valid Parentheses: all brackets matched → all() over close checks
  LC 125 — Valid Palindrome: any(c.isalnum() for c in s) — has valid chars?
  LC 242 — Valid Anagram: all(freq_s[c] == freq_t.get(c,0) for c in freq_s)
  LC 383 — Ransom Note: all(magazine has enough of each char)
  LC 424 — Longest Repeating Char Replacement: all chars in window satisfiable
  LC 567 — Permutation in String: all(window freqs match pattern freqs)
  LC 680 — Valid Palindrome II: all chars match after one deletion

SYNTAX:
  any(iterable)   → True if AT LEAST ONE element is truthy  (logical OR)
  all(iterable)   → True if ALL elements are truthy         (logical AND)
  both return False / True on empty per their identity:
    any([])  → False   (no truthy elements found)
    all([])  → True    (vacuous truth — nothing violated the condition)

VACUOUS TRUTH — all([]) is True:
  Think of all() as: "did anything FAIL the check?"
  Empty sequence → nothing failed → True.
  This matches math: ∀x ∈ {} : P(x) is vacuously true.
  Real-world: "all employees passed the test" when there are no employees → True.
  ⚠  Guard with: nums and all(x > 0 for x in nums)  if empty→False is needed.

SHORT-CIRCUIT BEHAVIOR:
  any() stops at the FIRST truthy value — remaining elements never evaluated.
  all() stops at the FIRST falsy value — remaining elements never evaluated.

  any(c.isdigit() for c in "abc3def"):
    'a'→False, 'b'→False, 'c'→False, '3'→True → STOP, return True
    "def" never checked.

  all(x > 0 for x in [1, 2, -1, 4]):
    1>0→True, 2>0→True, -1>0→False → STOP, return False
    4 never checked.

PASS GENERATOR — no extra list:
  ✅  any(c.isdigit() for c in s)    ← lazy, short-circuits
  ❌  any([c.isdigit() for c in s])  ← full list built before any() sees it

MANUAL LOOP EQUIVALENT — any() replaces boilerplate flag:
  # old way:
  found = False
  for c in s:
      if c.isdigit():
          found = True
          break

  # pythonic:
  found = any(c.isdigit() for c in s)

SLIDING WINDOW USE CASE (LC 567 pattern):
  # check if window freq matches pattern freq — all chars must match
  window_ok = all(window[c] == pattern[c] for c in pattern)

SLOW MOTION TRACE — all(x > 0 for x in [3, 1, -2, 5]):
  x=3:  3>0 → True  → continue
  x=1:  1>0 → True  → continue
  x=-2: -2>0 → False → STOP immediately, return False
  x=5:  never evaluated

KEY INSIGHT:
  any() = "does at least one pass?"  →  replaces found-flag loop
  all() = "does everything pass?"    →  replaces all-ok-flag loop
  Both short-circuit — faster than building a list first.

TIME / SPACE:
  Time:  O(n) worst case; O(1) best case with early exit
  Space: O(1) — generator feeds elements one at a time, no list allocated
```

In [ ]:
# Pattern 12: any() and all()
# Short-circuit aggregators — stop as soon as the answer is known.

# 1. basic any() / all()
nums = [3, 1, -2, 5, 0]
print(f"any negative  : {any(x < 0 for x in nums)}")    # True  — hits -2
print(f"all positive  : {all(x > 0 for x in nums)}")    # False — hits -2
print(f"any > 100     : {any(x > 100 for x in nums)}")  # False — scans all
print(f"all non-zero  : {all(x != 0 for x in nums)}")   # False — hits 0

# 2. empty iterable edge cases — know these cold
print(f"any([])       : {any([])}")   # False — no truthy element exists
print(f"all([])       : {all([])}")   # True  — nothing violated the condition

# 3. short-circuit — prove it fires early
def loud(x, label):
    print(f"  checking {label}={x}")
    return x > 0

print("any() short-circuit on [1, -1, 2]:")
result = any(loud(x, "x") for x in [1, -1, 2])   # stops at 1 (first True)
print(f"  result: {result}")

print("all() short-circuit on [1, -1, 2]:")
result = all(loud(x, "x") for x in [1, -1, 2])   # stops at -1 (first False)
print(f"  result: {result}")

# 4. any() replaces found-flag boilerplate
s = "hello3world"
# old way — flag variable
found_old = False
for c in s:
    if c.isdigit():
        found_old = True
        break
# pythonic
found_new = any(c.isdigit() for c in s)
print(f"has digit (old) : {found_old}")
print(f"has digit (new) : {found_new}")

# 5. sliding window condition check (LC 567 pattern)
from collections import Counter
pattern = "ab"
window  = "ba"
pat_freq = Counter(pattern)
win_freq = Counter(window)
window_ok = all(win_freq[c] == pat_freq[c] for c in pat_freq)
print(f"window matches pattern: {window_ok}")   # True


def ransom_note(ransomNote: str, magazine: str) -> bool:
    """
    LC 383 — Ransom Note
    Approach: all() over ransom chars — magazine must have enough of each.
    Args:
        ransomNote (str): letters needed.
        magazine (str): available letters.
    Returns:
        bool: True if ransomNote can be built from magazine letters.
    Time:  O(m + n) — two Counter builds + all() over unique ransom chars
    Space: O(1) — at most 26 keys each
    """
    from collections import Counter
    have = Counter(magazine)    # letter → count available
    need = Counter(ransomNote)  # letter → count required

    # slow motion on ransomNote="aa", magazine="aab":
    # have = {a:2, b:1}
    # need = {a:2}
    # all: 'a' → have['a']=2 >= need['a']=2 → True
    # all passed → True

    return all(have[c] >= need[c] for c in need)   # short-circuits on first shortage


def test_harness(fn):
    tests = [
        ("a",   "b",    False),   # 'a' not in magazine
        ("aa",  "ab",   False),   # need 2 'a', have 1
        ("aa",  "aab",  True),    # exact match
        ("",    "abc",  True),    # empty note — all([]) is True
        ("abc", "aabbcc", True),  # plenty available
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(*inputs)
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | input={inputs} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")


test_harness(ransom_note)

# guard pattern — empty list should return False, not vacuous True
data = []
safe = bool(data) and all(x > 0 for x in data)   # False (short-circuits at bool([]))
print(f"guarded all on [] : {safe}")

print("any_and_all defined.")